In [41]:
print("Whatsapp Chat Analyser")

Whatsapp Chat Analyser


In [42]:
import re
import pandas as pd
import warnings

warnings.filterwarnings("ignore")

In [43]:
def preprocess(data):

    pattern = r"\d{1,2}/\d{1,2}/\d{2,4},\s\d{1,2}:\d{2}(?:\s?[apAP][mM])?\s-\s"

    messages = re.split(pattern, data)[1:]
    dates = re.findall(pattern, data)

    df = pd.DataFrame({"user_message": messages, "message_date": dates})

    # CLEAN STEP (mandatory)
    df["message_date"] = df["message_date"].str.replace(" -", "", regex=False)

    # PARSE STEP
    df["message_date"] = pd.to_datetime(
        df["message_date"],
        dayfirst=True,
        errors="coerce",
    )

    df.rename(columns={"message_date": "date"}, inplace=True)

    users = []
    messages = []
    for message in df["user_message"]:
        entry = re.split("([\w\W]+?):\s", message)
        if entry[1:]:  # user name
            users.append(entry[1])
            messages.append(" ".join(entry[2:]))
        else:
            users.append("group_notification")
            messages.append(entry[0])

    df["user"] = users
    df["message"] = messages
    df.drop(columns=["user_message"], inplace=True)

    df["year"] = df["date"].dt.year
    df["month_num"] = df["date"].dt.month
    df["month"] = df["date"].dt.month_name()
    df["day"] = df["date"].dt.day
    df["day_name"] = df["date"].dt.day_name()
    df["hour"] = df["date"].dt.hour
    df["minute"] = df["date"].dt.minute
    df["only_date"] = df["date"].dt.date

    period = []
    for hour in df[["day_name", "hour"]]["hour"]:
        if hour == 23:
            period.append(str(hour) + "-" + str("00"))
        elif hour == 0:
            period.append(str("00") + "-" + str(hour + 1))
        else:
            period.append(str(hour) + "-" + str(hour + 1))

    df["period"] = period

    return df


# usage
with open("chats/WhatsApp Chat with HARDCORE STRIKERS 🇮🇳.txt", encoding="utf-8") as f:
    data = f.read()

df = preprocess(data)

In [44]:
df.sample(5)

,date,user,message,year,month_num,month,day,day_name,hour,minute,only_date,period
824,2026-03-22 11:38:00,Ankit Malik,PTT-20260322-WA0002.opus (file attached)\n,2026,3,March,22,Sunday,11,38,2026-03-22,11-12
643,2026-03-16 22:44:00,Shubam Bhai Gym,@⁨Ankit Malik⁩ Gand mardiye bhyaa ji\n,2026,3,March,16,Monday,22,44,2026-03-16,22-23
1906,2026-04-18 12:30:00,Pankaj Bhai Gym,<Media omitted>\n@⁨Shubam Bhai Gym⁩ check this...,2026,4,April,18,Saturday,12,30,2026-04-18,12-13
695,2026-03-20 20:49:00,kunal,@⁨Pankaj Bhai Gym⁩ app kha pr ho 🤣\n,2026,3,March,20,Friday,20,49,2026-03-20,20-21
1700,2026-04-13 13:55:00,Pankaj Bhai Gym,https://www.instagram.com/reel/DW_AmwDDi4p/?ig...,2026,4,April,13,Monday,13,55,2026-04-13,13-14


In [45]:
def detect_media(message):

    media_pattern = r"(IMG-\d+-WA\d+\.jpg|VID-\d+-WA\d+\.mp4|AUD-\d+-WA\d+\.opus|PTT-\d+-WA\d+\.opus)"

    if "<Media omitted>" in message:
        return "omitted"

    match = re.search(media_pattern, message)
    if match:
        return match.group()

    return None

In [46]:
df["media_file"] = df["message"].apply(detect_media)

# ✅ flags
df["is_media"] = df["media_file"].notnull()
df["is_omitted"] = df["media_file"] == "omitted"
df["has_actual_file"] = df["media_file"].notnull() & (df["media_file"] != "omitted")

In [47]:
df.sample(5)

,date,user,message,year,month_num,month,day,day_name,hour,minute,only_date,period,media_file,is_media,is_omitted,has_actual_file
1878,2026-04-17 13:05:00,Shubam Bhai Gym,Done\n,2026,4,April,17,Friday,13,5,2026-04-17,13-14,NaN,False,False,False
2402,2026-04-25 18:30:00,Shubam Bhai Gym,Punjabi agye oye🔥\n,2026,4,April,25,Saturday,18,30,2026-04-25,18-19,NaN,False,False,False
2102,2026-04-18 20:43:00,Pankaj Bhai Gym,Tu aur ankit 5 bje ground phuch jana\n,2026,4,April,18,Saturday,20,43,2026-04-18,20-21,NaN,False,False,False
2545,2026-04-26 18:47:00,Shubam Bhai Gym,CSK haar gyi kya😂\n,2026,4,April,26,Sunday,18,47,2026-04-26,18-19,NaN,False,False,False
2431,2026-04-25 18:42:00,Pankaj Bhai Gym,hm\n,2026,4,April,25,Saturday,18,42,2026-04-25,18-19,NaN,False,False,False


In [48]:
total_media = df["is_media"].sum()
omitted_media = df["is_omitted"].sum()
actual_media = df["has_actual_file"].sum()

In [49]:
total_media

np.int64(292)

In [50]:
omitted_media

np.int64(165)

In [51]:
actual_media

np.int64(127)

In [52]:
def get_media_type(file):

    # ✅ Handle NaN safely
    if pd.isna(file) or file == "omitted":
        return None

    if file.startswith("IMG"):
        return "image"
    elif file.startswith("VID"):
        return "video"
    elif file.startswith("AUD"):
        return "audio"
    elif file.startswith("PTT"):
        return "voice"

    return None

In [53]:
df["media_type"] = df["media_file"].apply(get_media_type)

In [54]:
df.sample(10)

,date,user,message,year,month_num,month,day,day_name,hour,minute,only_date,period,media_file,is_media,is_omitted,has_actual_file,media_type
1224,2026-04-01 13:01:00,Ankit Malik,Kunal Bhai Sahi baat hai yaar\n,2026,4,April,1,Wednesday,13,1,2026-04-01,13-14,NaN,False,False,False,NaN
1450,2026-04-08 00:37:00,kunal,🌳🌳\nLagao app\n,2026,4,April,8,Wednesday,0,37,2026-04-08,00-1,NaN,False,False,False,NaN
1408,2026-04-06 11:14:00,Shubam Bhai Gym,<Media omitted>\nLast 5 years 0 trophy 🤣\n,2026,4,April,6,Monday,11,14,2026-04-06,11-12,omitted,True,True,False,NaN
2155,2026-04-19 09:29:00,Pankaj Bhai Gym,Really Really sorry team for my today performa...,2026,4,April,19,Sunday,9,29,2026-04-19,9-10,NaN,False,False,False,NaN
2197,2026-04-22 10:31:00,Ankit Malik,https://www.instagram.com/reel/DXYsIbUTc0s/?ig...,2026,4,April,22,Wednesday,10,31,2026-04-22,10-11,NaN,False,False,False,NaN
2024,2026-04-18 13:55:00,Shubam Bhai Gym,Bhaii meri nahi sunega kya🤣\n,2026,4,April,18,Saturday,13,55,2026-04-18,13-14,NaN,False,False,False,NaN
1960,2026-04-18 13:36:00,Shubam Bhai Gym,Voting m ye konsa option hai\n,2026,4,April,18,Saturday,13,36,2026-04-18,13-14,NaN,False,False,False,NaN
1680,2026-04-13 13:12:00,Ankit Malik,Per caller wali banvaenge\n,2026,4,April,13,Monday,13,12,2026-04-13,13-14,NaN,False,False,False,NaN
689,2026-03-19 08:31:00,kunal,\n,2026,3,March,19,Thursday,8,31,2026-03-19,8-9,NaN,False,False,False,NaN
1874,2026-04-17 13:04:00,Shubam Bhai Gym,<Media omitted>\n,2026,4,April,17,Friday,13,4,2026-04-17,13-14,omitted,True,True,False,NaN


In [55]:
media_counts = df[df["has_actual_file"]]["media_type"].value_counts()

In [56]:
media_counts

media_type
voice    103
image     17
video      7
Name: count, dtype: int64

In [57]:
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

analyzer = SentimentIntensityAnalyzer()


def get_sentiment_label(score):
    if score >= 0.05:
        return "positive"
    elif score <= -0.05:
        return "negative"
    else:
        return "neutral"


def add_sentiment(df):
    df["sentiment_score"] = df["message"].apply(
        lambda x: analyzer.polarity_scores(x)["compound"]
    )

    df["sentiment"] = df["sentiment_score"].apply(get_sentiment_label)

    return df

In [59]:
df = add_sentiment(df)

In [60]:
df

,date,user,message,year,month_num,month,day,day_name,hour,minute,only_date,period,media_file,is_media,is_omitted,has_actual_file,media_type,sentiment_score,sentiment
0,2026-02-27 15:13:00,group_notification,Messages and calls are end-to-end encrypted. O...,2026,2,February,27,Friday,15,13,2026-02-27,15-16,NaN,False,False,False,NaN,0.2960,positive
1,2026-02-27 15:13:00,group_notification,"Abhijeet Bhaiya Gym created group ""Cricket Bro...",2026,2,February,27,Friday,15,13,2026-02-27,15-16,NaN,False,False,False,NaN,0.2500,positive
2,2026-02-27 15:13:00,group_notification,Abhijeet Bhaiya Gym added you\n,2026,2,February,27,Friday,15,13,2026-02-27,15-16,NaN,False,False,False,NaN,0.0000,neutral
3,2026-02-27 15:13:00,kunal,Hlo\n,2026,2,February,27,Friday,15,13,2026-02-27,15-16,NaN,False,False,False,NaN,0.0000,neutral
4,2026-02-27 15:13:00,Anshu Gujjar,💦💦\n,2026,2,February,27,Friday,15,13,2026-02-27,15-16,NaN,False,False,False,NaN,0.0000,neutral
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2617,2026-04-26 21:15:00,Yash Bhaiya Gym,Hahaha\n,2026,4,April,26,Sunday,21,15,2026-04-26,21-22,NaN,False,False,False,NaN,0.5574,positive
2618,2026-04-26 21:15:00,Yash Bhaiya Gym,Ok\n,2026,4,April,26,Sunday,21,15,2026-04-26,21-22,NaN,False,False,False,NaN,0.2960,positive
2619,2026-04-26 21:15:00,Yash Bhaiya Gym,Teri favourite to nhi hai na ye\n,2026,4,April,26,Sunday,21,15,2026-04-26,21-22,NaN,False,False,False,NaN,0.0000,neutral
2620,2026-04-26 21:16:00,Anshu Gujjar,Nhii usne hi bati thi\n,2026,4,April,26,Sunday,21,16,2026-04-26,21-22,NaN,False,False,False,NaN,0.0000,neutral


In [62]:
import plotly.express as px

fig = px.histogram(df, x="sentiment", title="Sentiment Distribution", color="sentiment")
fig.show()

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

def get_embeddings(messages):
    return model.encode(messages)

In [ ]:
texts = df["message"].tolist()
embeddings = get_embeddings(texts)

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np


def find_similar(query, df, embeddings, top_n=5):

    query_emb = model.encode([query])
    
    scores = cosine_similarity(query_emb, embeddings)[0]

    top_idx = np.argsort(scores)[-top_n:][::-1]

    return df.iloc[top_idx]["message"].tolist()

In [ ]:
query = input("Search similar messages: ")

if query:
    results = find_similar(query, df, embeddings)
    for r in results:
        print(r)

In [ ]:
def smart_reply(query, df, embeddings):

    query_emb = model.encode([query])
    
    scores = cosine_similarity(query_emb, embeddings)[0]

    best_idx = scores.argmax()

    return df.iloc[best_idx]["message"]

In [ ]:
msg = input("Type message: ")

if msg:
    reply = smart_reply(msg, df, embeddings)
    print(f"Suggested Reply: {reply}")

In [ ]:
bad_words = [
    # Original words
    "bc",
    "mc",
    "madarchod",
    "bhosdike",
    "chutiya",
    # Common Hindi/Hinglish abuses (A-Z coverage)
    "bhosad",
    "bhosdi",
    "bsdk",
    "bhen",
    "behen",
    "behenchod",
    "bkl",
    "bhadwe",
    "bhadwa",
    "chod",
    "chodu",
    "chut",
    "chutad",
    "chutiye",
    "chinal",
    "chinaal",
    "dalle",
    "dalal",
    "dharkan",
    "gandmare",
    "gandu",
    "gaandu",
    "gand",
    "gaand",
    "harami",
    "haramkhor",
    "hijra",
    "kutta",
    "kutte",
    "kamina",
    "kamine",
    "kamini",
    "kuttiya",
    "lodu",
    "lode",
    "laude",
    "lund",
    "lavde",
    "lawde",
    "mkc",
    "maaki",
    "maa ki",
    "maadar",
    "mother",
    "muth",
    "muthal",
    "randi",
    "rnd",
    "rand",
    "raand",
    "randiya" "sala",
    "saala",
    "suar",
    "suwar",
    "teri maa",
    "tere baap",
    "tatto",
    "tatton",
    "ullu",
    "ullu ke pathe",
    # Variations with spaces/symbols
    "m c",
    "b c",
    "m_c",
    "b_c",
    "mc_bc",
    "bc_mc",
    "bhen chod",
    "ma dar chod",
    "lun d",
    "ga and",
    "chu tiya",
    "ran di",
    "har ami",
    "gan du",
    # Leetspeak/number substitutions
    "b3hen",
    "ch0d",
    "g4nd",
    "l0de",
    "m@dar",
    "r@ndi",
    # Asterisk censored versions
    "b*c",
    "m*c",
    "ch*t",
    "l*nd",
    "g*nd",
    "r*ndi",
    "b***d",
    "bh***d",
    "madarch**",
    "behen***",
    # Other common toxic terms
    "bastard",
    "bitch",
    "damn",
    "hell",
    "shit",
    "fuck",
    "asshole",
    "idiot",
    "stupid",
    "fool",
    "loser",
    # Shortened/slang versions
    "bc",
    "mc",
    "wtf",
    "stfu",
    "gtfo",
    "mfr",
    "sob",
    "pos",
]


def detect_toxic(text):
    text = text.lower()
    return any(word in text for word in bad_words)

In [ ]:
df["toxic"] = df["message"].apply(detect_toxic)

In [ ]:
df['toxic'].value_counts()

In [ ]:
df[df['toxic'] == True]

In [ ]:
# import os
# import shutil

# SOURCE_FOLDER = "WhatsApp Chat with HARDCORE STRIKERS 🇮🇳"

# DEST_FOLDERS = {
#     "IMG": "chats/media/images",
#     "VID": "chats/media/videos",
#     "AUD": "chats/media/audio",
#     "PTT": "chats/media/voice",
# }

# # create folders if not exist
# for folder in DEST_FOLDERS.values():
#     os.makedirs(folder, exist_ok=True)

# # move files
# for file in os.listdir(SOURCE_FOLDER):
#     src_path = os.path.join(SOURCE_FOLDER, file)

#     if file.startswith("IMG"):
#         shutil.move(src_path, DEST_FOLDERS["IMG"])
#     elif file.startswith("VID"):
#         shutil.move(src_path, DEST_FOLDERS["VID"])
#     elif file.startswith("AUD"):
#         shutil.move(src_path, DEST_FOLDERS["AUD"])
#     elif file.startswith("PTT"):
#         shutil.move(src_path, DEST_FOLDERS["PTT"])

# print("✅ Media sorted successfully!")